# Twitch Analytics — Facteurs de variation d’audience et GTA VI
**Projet fil rouge · API Twitch Helix uniquement · collecte GitHub Actions pendant 7 jours**

**Problématique :** *Quels facteurs sont associés aux variations d’audience des streamers populaires sur Twitch (notoriété, jeu, horaire, durée du live) et comment la semaine de lancement de GTA VI modifie-t-elle les tendances observées ?*

**Périmètre :** jusqu’à **5 000 streams par relevé**, uniquement ceux ayant **au moins 1 000 spectateurs**, toutes les **4 heures**, sur **7 jours** (au plus 42 relevés). Le script GitHub Actions exécute **un relevé puis s’arrête** ; il est relancé au prochain créneau. Les collectes manquées ne sont pas reconstituées.

**Lecture :** les figures qui demandent plusieurs relevés sont indisponibles tant que l’historique est insuffisant. Ce notebook étudie des **associations**, pas des causes.

## 1. Import du CSV et préparation

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display

FICHIER = Path('twitch_helix_7jours.csv')
# Le fichier test peut servir à vérifier le notebook avant la semaine complète.
# Exemple : FICHIER = Path('twitch_helix_test_10min.csv')
if not FICHIER.exists():
    raise FileNotFoundError(f'CSV introuvable : {FICHIER}. Place-le près du notebook.')

brut = pd.read_csv(FICHIER, encoding='utf-8-sig', low_memory=False)
print(f'{len(brut):,} lignes, {len(brut.columns)} colonnes — {FICHIER.name}'.replace(',', ' '))
display(brut.head(3))

requis = ['stream_id', 'user_id', 'game_name', 'viewer_count', 'collected_at']
manquants = [c for c in requis if c not in brut.columns]
if manquants:
    raise ValueError(f'Colonnes requises absentes : {manquants}')

df = brut.copy()
if 'scheduled_at' not in df: df['scheduled_at'] = df['collected_at']
for c in ['follower_count','follower_status','broadcaster_type','stream_duration_minutes',
          'viewer_follower_ratio','started_at','language','user_name','title']:
    if c not in df: df[c] = np.nan
for c in ['scheduled_at','collected_at','started_at']:
    df[c] = pd.to_datetime(df[c], utc=True, errors='coerce')
for c in ['viewer_count','follower_count','stream_duration_minutes']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['slot'] = df['scheduled_at'].fillna(df['collected_at'])
df['game_name'] = df['game_name'].fillna('Catégorie inconnue').astype(str).str.strip()
df['broadcaster_type'] = df['broadcaster_type'].fillna('inconnu').replace('', 'inconnu')

df['heure_paris'] = df['slot'].dt.tz_convert('Europe/Paris').dt.hour
df['jour_paris'] = df['slot'].dt.tz_convert('Europe/Paris').dt.strftime('%A')
if df['stream_duration_minutes'].notna().sum() == 0:
    df['stream_duration_minutes'] = (df['collected_at'] - df['started_at']).dt.total_seconds()/60

nom = df['game_name'].str.casefold()
df['gta'] = np.select([
    nom.str.contains(r'grand theft auto vi(?:\b|$)|grand theft auto 6(?:\b|$)|^gta vi$|^gta 6$',regex=True),
    nom.str.contains(r'grand theft auto v(?:\b|$)|grand theft auto 5(?:\b|$)|^gta v$|^gta 5$',regex=True)
], ['GTA VI','GTA V'], default='Autres')

invalides = df['viewer_count'].isna() | (df['viewer_count'] < 1000) | df['slot'].isna() | df['stream_id'].isna()
nb_invalides = int(invalides.sum())
df = df.loc[~invalides].copy()
nb_doublons = int(df.duplicated(['slot','stream_id']).sum())
df = df.drop_duplicates(['slot','stream_id']).sort_values(['slot','viewer_count'],ascending=[True,False])

plt.rcParams.update({'figure.figsize': (10, 4.6),'axes.grid': True,'grid.alpha': .18})
fmt = FuncFormatter(lambda x,pos: f'{x:,.0f}'.replace(',', ' '))
def fini(ax, axe='y'):
    (ax.yaxis if axe=='y' else ax.xaxis).set_major_formatter(fmt)
    plt.tight_layout(); plt.show()
def absent(raison):
    print('Graphique non disponible : ' + raison)

## 2. Audit : qualité, quantité et limites

In [ ]:
qualite = pd.DataFrame({'manquants':brut.isna().sum(),'% manquants':brut.isna().mean().mul(100).round(1)})
display(qualite.sort_values('% manquants',ascending=False))
releves = df.groupby('slot').agg(streams=('stream_id','nunique'), streamers=('user_id','nunique'), audience_totale=('viewer_count','sum'))
resume = pd.Series({
    'Observations valides': len(df), 'Relevés distincts': len(releves),
    'Chaînes distinctes': df.user_id.nunique(), 'Jeux distincts':df.game_name.nunique(),
    'Rejets (seuil/date/identifiant)': nb_invalides, 'Doublons par relevé':nb_doublons,
    'Audience minimale':df.viewer_count.min(), 'Audience médiane':df.viewer_count.median(),
    'Followers renseignés (%)':round(100*df.follower_count.notna().mean(),1) if len(df) else 0,
    'Début UTC':df.slot.min(), 'Fin UTC':df.slot.max(),
})
display(resume.to_frame('Résultat'))
if len(releves)>1:
    intervalles=releves.index.to_series().diff().dropna().dt.total_seconds()/3600
    print('Intervalle médian entre relevés (h) :',round(intervalles.median(),2))
else: print('Historique encore insuffisant : un seul relevé.')
print('Rappel : échantillon tronqué aux streams ≥ 1 000 viewers ; ne représente pas Twitch entier.')

## 3. Distribution des audiences
**But :** visualiser les fortes disparités. Le zoom à 95 % ne retire aucune ligne du jeu de données.

In [ ]:
if df.empty: absent('aucune observation')
else:
    limite=max(df.viewer_count.quantile(.95),1000)
    fig,ax=plt.subplots()
    ax.hist(df.loc[df.viewer_count<=limite,'viewer_count'],bins=35,edgecolor='white')
    ax.set(xlabel='Spectateurs (zoom au 95e percentile)',ylabel='Nombre de relevés de streams',title='Distribution des audiences des streams populaires')
    fini(ax,'x')
    print('Audience maximale (conservée dans le CSV) :',f'{df.viewer_count.max():,.0f}'.replace(',',' '))

## 4. Notoriété : followers et audience
**But :** mesurer l’association entre followers et audience. **Un point = une chaîne** (médianes), pour éviter qu’un streamer souvent observé compte plusieurs fois.

In [ ]:
notoriete = df[df.follower_count.gt(0)].groupby('user_id').agg(followers=('follower_count','median'),audience=('viewer_count','median')).dropna()
if len(notoriete)<5: absent('au moins 5 chaînes ayant des followers renseignés nécessaires')
else:
    fig,ax=plt.subplots()
    ax.scatter(notoriete.followers,notoriete.audience,alpha=.55,s=22)
    ax.set(xlabel='Nombre de followers (par chaîne)',ylabel='Audience médiane observée',title='Followers et spectateurs : relation observée')
    fini(ax,'x')
    print('Chaînes :',len(notoriete),'— corrélation Spearman :',round(notoriete.followers.corr(notoriete.audience,method='spearman'),3))

## 5. Audience par tranche de notoriété
**But :** comparer les audiences médianes des chaînes selon leur taille, sans supposer un lien de causalité.

In [ ]:
if notoriete.empty: absent('followers non disponibles')
else:
    bins=[0,50_000,500_000,float('inf')]
    labels=['< 50 000','50 000–500 000','> 500 000']
    tmp=notoriete.copy()
    tmp['tranche']=pd.cut(tmp.followers,bins=bins,labels=labels,include_lowest=True)
    med=tmp.groupby('tranche',observed=True).audience.median().reindex(labels).dropna()
    if med.empty: absent('pas de données par tranche')
    else:
        fig,ax=plt.subplots()
        ax.bar(med.index.astype(str),med.values)
        ax.set(xlabel='Followers',ylabel='Audience médiane par chaîne',title='Audience selon la tranche de notoriété')
        fini(ax)
        display(tmp.groupby('tranche',observed=True).size().rename('Chaînes par tranche').to_frame())

## 6. Statut de la chaîne et durée du live
Deux facteurs possibles, à interpréter avec prudence.

In [ ]:
valides=df[df.broadcaster_type.isin(['partner','affiliate'])]
if valides.empty: absent('statut Partner / Affiliate indisponible')
else:
    med=valides.groupby('broadcaster_type').viewer_count.median()
    fig,ax=plt.subplots()
    ax.bar(med.index,med.values)
    ax.set(title='Audience médiane selon le statut',xlabel='Statut',ylabel='Spectateurs')
    fini(ax)

valides=df[df.stream_duration_minutes.between(0,24*60) & df.viewer_count.notna()]
if len(valides)<10: absent('durée de live insuffisamment renseignée')
else:
    fig,ax=plt.subplots()
    ax.scatter(valides.stream_duration_minutes,valides.viewer_count.clip(upper=valides.viewer_count.quantile(.95)),s=10,alpha=.22)
    ax.set(title='Durée du live et audience (zoom visuel 95 %)',xlabel='Durée du live (minutes)',ylabel='Spectateurs (plafond visuel)')
    fini(ax)

## 7. Horaires et évolution des audiences
**Attention :** les horaires sont ceux des **relevés** en heure de Paris, pas nécessairement le début des streams. La composition de l’échantillon évolue entre relevés.

In [ ]:
if len(releves)<2: absent('au moins deux collectes nécessaires')
else:
    par_heure=df.groupby('heure_paris').viewer_count.median()
    fig,ax=plt.subplots()
    ax.plot(par_heure.index,par_heure.values,marker='o')
    ax.set(xlabel='Heure du relevé (Paris)',ylabel='Audience médiane des streams observés',title='Audiences observées selon le créneau horaire',xticks=par_heure.index)
    fini(ax)

    serie=df.groupby('slot').viewer_count.median().sort_index()
    fig,ax=plt.subplots(figsize=(11,4))
    ax.plot(serie.index,serie.values,marker='o',markersize=4)
    ax.set(xlabel='Date du relevé (UTC)',ylabel='Audience médiane',title='Évolution de l’audience médiane des streams sélectionnés')
    fini(ax)

## 8. Évolution des **mêmes streamers** entre deux relevés
**But :** éviter de confondre variation réelle d’une chaîne et arrivée/disparition de chaînes dans l’échantillon. Ne conserver que deux observations successives **du même live**.

In [ ]:
if len(releves)<2: absent('plusieurs relevés nécessaires')
else:
    panel=df.sort_values(['stream_id','slot']).copy()
    panel['audience_precedente']=panel.groupby('stream_id').viewer_count.shift(1)
    panel['slot_precedent']=panel.groupby('stream_id').slot.shift(1)
    panel['ecart_heures']=(panel.slot-panel.slot_precedent).dt.total_seconds()/3600
    panel['variation_pct']=100*(panel.viewer_count-panel.audience_precedente)/panel.audience_precedente
    stable=panel[panel.audience_precedente.gt(0)&panel.ecart_heures.between(.5,6)].copy()
    if len(stable)<5: absent('moins de 5 variations valides entre deux relevés du même live')
    else:
        plaf=max(10,stable.variation_pct.abs().quantile(.95))
        fig,ax=plt.subplots()
        ax.hist(stable.variation_pct.clip(-plaf,plaf),bins=30,edgecolor='white')
        ax.set(xlabel='Variation de spectateurs (%) — extrêmes plafonnés visuellement',ylabel='Observations appariées',title='Variations d’audience des mêmes lives')
        fini(ax,'x')
        print('Paires de relevés du même live :',len(stable),'— variation médiane :',round(stable.variation_pct.median(),1),'%')

## 9. GTA VI : GTA V vs GTA VI au fil des relevés
**Contexte :** sortie officielle de GTA VI le **19 novembre 2026**. Le graphique compare les jeux **dans l’échantillon des streams ≥ 1 000 spectateurs**. La somme des audiences est une photographie par relevé, pas un total de spectateurs uniques de la semaine.

In [ ]:
gta=df[df.gta.isin(['GTA V','GTA VI'])]
if gta.empty: absent('aucun stream GTA V / GTA VI')
else:
    serie=gta.pivot_table(index='slot',columns='gta',values='viewer_count',aggfunc='sum',fill_value=0)
    for cat in ['GTA V','GTA VI']:
        if cat not in serie:serie[cat]=0
    fig,ax=plt.subplots(figsize=(11,4))
    for cat in ['GTA V','GTA VI']:ax.plot(serie.index,serie[cat],marker='o',label=cat)
    ax.set(xlabel='Date du relevé (UTC)',ylabel='Somme des spectateurs observés',title='GTA V vs GTA VI — audience des streams sélectionnés')
    ax.legend();fini(ax)
    if len(serie)<2: print('Un seul relevé : évolution temporelle non encore observable.')

## 10. GTA VI : part d’audience et nombre de chaînes
**But :** étudier la visibilité de GTA VI parmi les streams populaires suivis. La part est relative à **cet échantillon filtré**, et non à toute la plateforme.

In [ ]:
if len(releves)<2: absent('au moins deux relevés nécessaires')
else:
    total=df.groupby('slot').viewer_count.sum()
    gta6=df[df.gta.eq('GTA VI')].groupby('slot').viewer_count.sum().reindex(total.index,fill_value=0)
    part=100*gta6/total
    fig,ax=plt.subplots(figsize=(11,4))
    ax.plot(part.index,part.values,marker='o')
    ax.set(title='Part d’audience GTA VI parmi les streams observés',ylabel='Part (%)',xlabel='Relevé (UTC)',ylim=(0,100))
    plt.tight_layout();plt.show()

    nb=df[df.gta.eq('GTA VI')].groupby('slot').user_id.nunique().reindex(total.index,fill_value=0)
    fig,ax=plt.subplots(figsize=(11,3.7))
    ax.plot(nb.index,nb.values,marker='o')
    ax.set(title='Chaînes GTA VI au-dessus de 1 000 spectateurs',ylabel='Nombre de chaînes',xlabel='Relevé (UTC)')
    fini(ax)

## 11. GTA VI face aux autres jeux et éventuels changements GTA V → VI
**Limite :** une chaîne observée sur GTA V puis GTA VI ne prouve pas que le changement de jeu a causé une variation d’audience.

In [ ]:
if df.empty: absent('aucune observation')
else:
    # Une mesure moyenne par catégorie et par créneau évite de privilégier les créneaux répétés.
    top=df.groupby(['slot','game_name']).viewer_count.sum().groupby('game_name').median().nlargest(10).sort_values()
    fig,ax=plt.subplots(figsize=(10,5))
    ax.barh(top.index,top.values)
    ax.set(title='10 jeux — audience médiane totale par relevé',xlabel='Spectateurs par relevé',ylabel='Jeu')
    fini(ax,'x')

gta=df[df.gta.isin(['GTA V','GTA VI'])].sort_values('slot')
changements=[]
for uid,subset in gta.groupby('user_id'):
    v=subset[subset.gta.eq('GTA V')]
    vi=subset[subset.gta.eq('GTA VI')]
    if not v.empty and not vi.empty and v.slot.min()<vi.slot.max():
        changements.append({'chaine':str(subset.user_name.iloc[0]),'GTA V':v.viewer_count.median(),'GTA VI':vi.viewer_count.median()})
if not changements: absent('pas de chaîne observée sur GTA V puis sur GTA VI')
else:
    transitions=pd.DataFrame(changements).sort_values('GTA VI',ascending=False).head(10).set_index('chaine')
    fig,ax=plt.subplots(figsize=(11,5))
    transitions.plot.bar(ax=ax,rot=30)
    ax.set(title='Mêmes chaînes observées sur GTA V puis GTA VI',ylabel='Audience médiane des relevés',xlabel='Chaîne')
    fini(ax)

## 12. Conclusion pour l’oral

- **Ce qui est observé :** audiences des streams populaires (seuil ≥ 1 000), notoriété mesurée par `follower_count`, catégorie, horaire, durée et statut.
- **Ce qui peut être étudié après plusieurs relevés :** évolution des mêmes lives, différences entre créneaux, place de GTA VI dans la semaine de lancement.
- **Nettoyage :** vérifier dates, seuil, doublons `(slot, stream_id)`, valeurs manquantes et qualité des followers ; conserver les grosses audiences sans les confondre avec des erreurs.
- **Biais :** exclusion des petits streams, relevés espacés de 4 h, échantillon qui change de composition, une semaine seulement, pas de preuve de causalité.
- **Architecture cible :** Helix → GitHub Actions → CSV horodaté → ETL / PostgreSQL → dashboards Streamlit → prototype de prédiction et monitoring.
- **Suite :** prolonger la collecte pour améliorer le Machine Learning, avec une validation chronologique et comparaison à un modèle simple.

**Important :** ne pas attribuer à GTA VI tous les mouvements d’audience sans données comparables avant/après et sans autres facteurs de contrôle.